In [ ]:
import getpass
import os
from dotenv import load_dotenv

load_dotenv('../.env')
openai_api_key = os.getenv('OPENAI_ACCESS_KEY')
huggingface_access_key = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [ ]:
from langchain_elasticsearch import ElasticsearchStore
from langchain_community.embeddings import HuggingFaceEmbeddings

from typing import Any, Dict, Iterable

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from langchain.embeddings import DeterministicFakeEmbedding
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_elasticsearch import ElasticsearchRetriever
from sentence_transformers import SentenceTransformer

In [ ]:
ElasticsearchRetriever

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

class SentenceEmbedder:
    def __init__(self, model_name: str = 'dmlls/all-mpnet-base-v2-negation'):
        # Load model and tokenizer from HuggingFace Hub
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        # Mean Pooling - Take attention mask into account for correct averaging
        token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def get_embeddings(self, sentence: str) -> list:
        # Tokenize sentence
        encoded_input = self.tokenizer([sentence], padding=True, truncation=True, return_tensors='pt')

        # Compute token embeddings
        with torch.no_grad():
            model_output = self.model(**encoded_input)

        # Perform pooling
        sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])

        # Normalize embeddings
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

        # Convert to list and return
        return sentence_embeddings[0].tolist()


embedder = SentenceEmbedder()



In [ ]:
es_url = "https://localhost:9200"
username = "elastic"
password = "QQ7wWoB_WnKe*L*X9tAW"


es_client = Elasticsearch(
    hosts=[es_url],
    ca_certs="./ca.crt",
    basic_auth=(username, password),
    max_retries=10,
)

# Test the connection
info = es_client.info()
print(info)


In [ ]:
from elasticsearch import Elasticsearch

def create_index(es_client: Elasticsearch, index_name: str, vector_dims: int):
    es_client.indices.create(
        index=index_name,
        body={
            "settings": {
                "analysis": {
                    "analyzer": {
                        "standard_lowercase": {
                            "type": "custom",
                            "tokenizer": "standard",
                            "filter": ["lowercase"]
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "criterion": {
                        "type": "text",
                        "analyzer": "standard_lowercase"
                    },
                    "criterion_vector": {  # Renamed and moved to be a separate top-level field
                        "type": "dense_vector",
                        "dims": vector_dims
                    },
                    "entities": {
                        "type": "nested",
                        "properties": {
                            "normalized_id": {"type": "keyword"},
                            "synonyms": {"type": "text", "analyzer": "standard_lowercase"},
                            "is_negated": {"type": "keyword"},
                            "entity": {"type": "text", "analyzer": "standard_lowercase"},
                            "class": {"type": "keyword"}
                        }
                    },
                    
                    "nct_id": {"type": "keyword", "index": False}  # Stored but not indexed
                }
            }
        },
    )


In [ ]:
import os
import json

def prepare_documents(folder_path):
    documents = []
    # Loop through all files in the specified directory
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):  # Check for JSON files
            file_path = os.path.join(folder_path, filename)
            with open(file_path, 'r') as file:
                data = json.load(file)  # Load JSON content
                nct_id = data['nct_id']  # Extract the NCT ID
                for criterion in data['criteria']:  # Iterate through each criterion
                    # Create a document for Elasticsearch
                    document = {
                        'criterion': criterion['criterion'],
                        'entities': criterion['entities'],
                        'eligibility_type': criterion['type'],
                        'nct_id': nct_id  # Include metadata containing NCT ID
                        
                    }
                    documents.append(document)
    return documents

# Example usage
folder_path = '../../data/ner_trial/'
documents = prepare_documents(folder_path)
print(f"Prepared {len(documents)} documents for indexing.")


In [ ]:
# Apply the function
number_of_indexed_documents = index_data(
    es_client=es_client,
    index_name="clinicaltrials",
    embedder=embedder,
    documents=documents,
    vector_dims=768,
)

In [ ]:
def count_documents(es_client, index_name):
    try:
        count = es_client.count(index=index_name)
        print(f"Total documents in the index '{index_name}': {count['count']}")
        return count['count']
    except Exception as e:
        print(f"Error counting documents in index '{index_name}': {str(e)}")
        return None


In [ ]:
count_documents(es_client, "eligibility_criteria")

In [ ]:
def vector_query(search_query: str) -> Dict:
    vector = embedder.get_embeddings(search_query)  # same embeddings as for indexing
    return {
        "size": 5000,  # return top 10 results
        "min_score": 0.0,
        "knn": {
            "field": "criterion_vector",
            "query_vector": vector,
            "num_candidates": 10000,
        }
    }

vector_retriever = ElasticsearchRetriever(
    es_client=es_client,
    index_name="eligibility_criteria",
    body_func=vector_query,
    content_field="criterion"
)


documents = vector_retriever.invoke("EGFR")

In [ ]:
dict(documents[0]).keys()

In [ ]:
for doc in documents:
    print(doc.page_content)

In [ ]:
text_field = "criterion"
index_name = "eligibility_criteria"
from elasticsearch import Elasticsearch


def bm25_query(search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
    return {
        "size": 8000,  # Controls the number of search results returned
        "query": {
            "bool": {
                "must": [
                    {
                        "match": {
                            "eligibility_type.keyword": eligibility_type
                        }
                    },
                    {
                        "bool": {
                            "should": [
                                {
                                    "match": {
                                        "criterion": {
                                            "query": search_query,
                                            # "fuzziness": "AUTO"
                                        }
                                    }
                                },
                                {
                                    "nested": {
                                        "path": "entities",
                                        "query": {
                                            "bool": {
                                                "should": [
                                                    {
                                                        "match": {
                                                            "entities.entity": {
                                                                "query": search_query,
                                                                # "fuzziness": "AUTO"
                                                            }
                                                        }
                                                    },
                                                    {
                                                        "match": {
                                                            "entities.synonyms": {
                                                                "query": search_query,
                                                                # "fuzziness": "AUTO"
                                                            }
                                                        }
                                                    }
                                                ]
                                            }
                                        }
                                    }
                                }
                            ]
                        }
                    }
                ]
            }
        },
    }
    
bm25_retriever = ElasticsearchRetriever(
    es_client=es_client,
    index_name=index_name,
    body_func=bm25_query,
    content_field=text_field,
)

# bm25_retriever.invoke("KRAS")

In [ ]:
documents_result = bm25_retriever.invoke("Metastatic treatment") 

In [ ]:
for doc in documents_result:
    print(doc.page_content)

In [ ]:
len(documents_result)

In [ ]:
from typing import List
import logging
logger = logging.getLogger(__name__)

index_name = "eligibility_criteria"
text_field = "criterion"

def combined_query(search_queries: List[str], eligibility_type: str = "Inclusion Criteria") -> Dict:
        should_clauses = []
        for search_query in search_queries:
            vector = embedder.get_embeddings(search_query)
            should_clauses.extend([
                # {"match": {"criterion": {"query": search_query, "fuzziness": "AUTO"}}},
                # {
                #     "nested": {
                #         "path": "entities",
                #         "query": {
                #             "bool": {
                #                 "should": [
                #                     {"match": {"entities.synonyms": {"query": search_query, "fuzziness": "AUTO"}}}
                #                 ]
                #             }
                #         }
                #     }
                # },
            {
            "knn": {
                "field": "criterion_vector",
                "query_vector": vector,
                "num_candidates": 10000  # Number of candidates to consider for k-NN
            }
            }
            ])

        logger.info("Constructed combined query for queries: %s", search_queries)
        return {
            "size": 80,
            "query": {
                "bool": {
                    "must": [
                        {"match": {"eligibility_type.keyword": eligibility_type}}
                    ],
                    "should": should_clauses,
                    "minimum_should_match": 1  # At least one should clause should match
                }
            },
            "explain": "true"
        }
        
combined_retriever = ElasticsearchRetriever(
    es_client=es_client,
    index_name=index_name,
    body_func=combined_query,
    content_field=text_field,
)

In [ ]:
documents = combined_retriever.invoke(["Adjuvant treatment"])

In [ ]:
for doc in documents[7500:7550]:
    print(doc.metadata["_score"])

In [ ]:
documents[10]

In [ ]:
for doc in documents:
    print(doc.page_content)

In [ ]:
documents_result[0].metadata['_source']["nct_id"]

In [ ]:
from langchain.retrievers import EnsembleRetriever
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], weights=[0.4, 0.6]
)

In [ ]:
docs = ensemble_retriever.invoke("KRAS mutation")
docs

In [ ]:
len(docs)

In [ ]:
text_field = "criterion"
index_name = "clinicaltrials"
def hybrid_query(search_query: str) -> Dict:
    vector = embeddings.embed_query(search_query)  # same embeddings as for indexing
    return {
        "size": 500,  # Controls the number of search results returned
                "query": {
            "match": {
               text_field: {
                    "query": search_query,
                    "fuzziness": "AUTO",
                }
            },
        },
        "knn": {
            "field": "criterion_vector",
            "query_vector": vector,
            "k": 500,
        },
        "rank": {"rrf": {"window_size": 1000}},
    }


hybrid_retriever = ElasticsearchRetriever.from_es_params(
    index_name=index_name,
    body_func=hybrid_query,
    content_field=text_field,
    url=es_url,
    username=username,
    password=password,
)

docs = hybrid_retriever.invoke("KRAS mutaion")


text_field = "criterion"
vector_retriever = ElasticsearchRetriever.from_es_params(
    index_name="clinicaltrials",
    body_func=vector_query,
    content_field=text_field,
    url=es_url,
    username=username,
    password=password,
)

vector_retriever.invoke("Retinal vein occlusion")

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

sentences = "The patient has a history of retinal vein occlusion.",
    # "The patient does have a history of retinal vein occlusion.",


model = SentenceTransformer(model_name_or_path="dmlls/all-mpnet-base-v2-negation")
len(model.encode(sentences).tolist()[0])

# def cosine_similarity(v1, v2):
#     return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [ ]:
embeddings[0].tolist()

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

class SentenceEmbedder:
    def __init__(self, model_name: str = 'dmlls/all-mpnet-base-v2-negation'):
        # Load model and tokenizer from HuggingFace Hub
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):
        # Mean Pooling - Take attention mask into account for correct averaging
        token_embeddings = model_output[0]  # First element of model_output contains all token embeddings
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def get_embeddings(self, sentence: str) -> list:
        # Tokenize sentence
        encoded_input = self.tokenizer([sentence], padding=True, truncation=True, return_tensors='pt')

        # Compute token embeddings
        with torch.no_grad():
            model_output = self.model(**encoded_input)

        # Perform pooling
        sentence_embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])

        # Normalize embeddings
        sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

        # Convert to list and return
        return sentence_embeddings[0].tolist()

# Example usage
sentence = documents[0]['criterion']
embedder = SentenceEmbedder()
embedding = embedder.get_embeddings(sentence)
print(embedding)


In [ ]:
embedding = embedder.get_embeddings("hello")



In [ ]:
documents[0]['criterion']

In [ ]:
print(embedding)

In [ ]:
from typing import List, Dict
import logging
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from elasticsearch import Elasticsearch
import nest_asyncio
import asyncio
from FlagEmbedding import FlagReranker

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

class ThreeStageSearch:
    def __init__(self, es_client: Elasticsearch, embedder) -> None:
        self.es_client = es_client
        self.embedder = embedder
        self.cross_encoder = self.load_cross_encoder()
        self.index_name = "eligibility_criteria"
        self.text_field = "criterion"

    def load_cross_encoder(self) -> "CrossEncoder":
        class CrossEncoder:
            def __init__(self) -> None:
                self.device = (
                    torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
                )
                logging.info(f"Using device: {self.device}")
                model_name = "BAAI/bge-reranker-large"
                self.tokenizer = AutoTokenizer.from_pretrained(model_name)
                self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
                self.model = self.model.to(self.device)

            async def rerank_pair(self, pair: List[str]) -> float:
                with torch.inference_mode():
                    inputs = self.tokenizer(
                        [pair],
                        padding=True,
                        truncation=True,
                        return_tensors="pt",
                        max_length=512,
                    )
                    inputs = inputs.to(self.device)
                    score = (
                        self.model(**inputs, return_dict=True)
                        .logits.view(
                            -1,
                        )
                        .float()
                    )
                    return score.item()

            async def rerank(self, pairs: List[List[str]]) -> List[float]:
                tasks = [self.rerank_pair(pair) for pair in pairs]
                return await asyncio.gather(*tasks)

        return CrossEncoder()

    def bm25_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
        return {
            "size": 8000,
            "query": {
                "bool": {
                    "must": [
                        {
                            "match": {
                                "eligibility_type.keyword": eligibility_type
                            }
                        },
                        {
                            "bool": {
                                "should": [
                                    {
                                        "match": {
                                            "criterion": {
                                                "query": search_query,
                                                "fuzziness": "AUTO"
                                            }
                                        }
                                    },
                                    {
                                        "nested": {
                                            "path": "entities",
                                            "query": {
                                                "bool": {
                                                    "should": [
                                                        {
                                                            "match": {
                                                                "entities.entity": {
                                                                    "query": search_query,
                                                                    "fuzziness": "AUTO"
                                                                }
                                                            }
                                                        },
                                                        {
                                                            "match": {
                                                                "entities.synonyms": {
                                                                    "query": search_query,
                                                                    "fuzziness": "AUTO"
                                                                }
                                                            }
                                                        }
                                                    ]
                                                }
                                            }
                                        }
                                    }
                                ]
                            }
                        }
                    ]
                }
            },
        }
        
    def vector_query(self, search_query: str, document_ids: List[str], min_score: float = 0.5) -> Dict:
        vector = self.embedder.get_embeddings(search_query)  # same embeddings as for indexing
        return {
            "size": 5000,  # return top 500 results
            "min_score": min_score,
            "query": {
                "bool": {
                    "must": [
                        {
                            "terms": {
                                "_id": document_ids
                            }
                        },
                        {
                            "knn": {
                                "field": "criterion_vector",
                                "query_vector": vector,
                                "num_candidates": 10000
                            }
                        }
                    ]
                }
            }
        }
        
    async def retrieve(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> List[Dict]:
        # Stage 1: BM25 Search
        bm25_retriever = ElasticsearchRetriever(
            es_client=self.es_client,
            index_name=self.index_name,
            body_func=self.bm25_query,
            content_field=self.text_field,
        )
        bm25_documents = bm25_retriever.invoke(search_query)
        bm25_document_ids = [doc.metadata['_id'] for doc in bm25_documents]
        
        # Stage 2: Semantic Search on BM25 Results
        vector_retriever = ElasticsearchRetriever(
            es_client=self.es_client,
            index_name=self.index_name,
            body_func=lambda query: self.vector_query(query, bm25_document_ids),
            content_field=self.text_field
        )
        vector_documents = vector_retriever.invoke(search_query)
        
        # Stage 3: Neural Re-ranking with async processing
        pairs = [[search_query, doc.page_content] for doc in vector_documents]
        scores = await self.cross_encoder.rerank(pairs)
        
        # Attach scores to documents and sort by scores
        for i, doc in enumerate(vector_documents):
            doc.metadata['score'] = scores[i]
        
        # Filter out documents with non-positive scores
        positive_documents = [doc for doc in vector_documents if doc.metadata['score'] > 0]
        sorted_documents = sorted(positive_documents, key=lambda x: x.metadata['score'], reverse=True)
        
        return sorted_documents

# Initialize ThreeStageSearch class
three_stage_search = ThreeStageSearch(es_client=es_client, embedder=embedder)

# Perform search
search_query = "KRAS mutation"

# Run the asynchronous function
documents = await three_stage_search.retrieve(search_query)

# Print the documents
for doc in documents:
    print(doc.page_content)


In [ ]:
len(documents)

In [ ]:
from FlagEmbedding import FlagReranker
reranker = FlagReranker('BAAI/bge-reranker-large', use_fp16=True, device="cuda:4") # Setting use_fp16 to True speeds up computation with a slight performance degradation

score = reranker.compute_score(['query', 'passage'])
print(score)

scores = reranker.compute_score([['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']])
print(scores)


In [ ]:
from typing import List, Dict
import logging
import torch
from sagemaker_inference import encoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from elasticsearch import Elasticsearch
from FlagEmbedding import FlagReranker

class ThreeStageSearch:
    def __init__(self, es_client: Elasticsearch, embedder) -> None:
        self.es_client = es_client
        self.embedder = embedder
        self.reranker = self.load_reranker()
        self.index_name = "eligibility_criteria"
        self.text_field = "criterion"

    def load_reranker(self) -> "FlagReranker":
        from FlagEmbedding import FlagReranker
        return FlagReranker('BAAI/bge-reranker-large', use_fp16=True, device="cuda:4")

    def bm25_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
        return {
            "size": 8000,
            "query": {
                "bool": {
                    "must": [
                        {
                            "match": {
                                "eligibility_type.keyword": eligibility_type
                            }
                        },
                        {
                            "bool": {
                                "should": [
                                    {
                                        "match": {
                                            "criterion": {
                                                "query": search_query,
                                                "fuzziness": "AUTO"
                                            }
                                        }
                                    },
                                    {
                                        "nested": {
                                            "path": "entities",
                                            "query": {
                                                "bool": {
                                                    "should": [
                                                        {
                                                            "match": {
                                                                "entities.entity": {
                                                                    "query": search_query,
                                                                    "fuzziness": "AUTO"
                                                                }
                                                            }
                                                        },
                                                        {
                                                            "match": {
                                                                "entities.synonyms": {
                                                                    "query": search_query,
                                                                    "fuzziness": "AUTO"
                                                                }
                                                            }
                                                        }
                                                    ]
                                                }
                                            }
                                        }
                                    }
                                ]
                            }
                        }
                    ]
                }
            },
        }

    def vector_query(self, search_query: str, min_score: float = 0.0) -> Dict:
        vector = self.embedder.get_embeddings(search_query)  # same embeddings as for indexing
        return {
            "size": 5000,  # return top 500 results
            "min_score": min_score,
            "knn": {
                "field": "criterion_vector",
                "query_vector": vector,
                "num_candidates": 10000
            }
        }

    def retrieve(self, search_query: str, eligibility_type: str = "Exclusion Criteria") -> List[Dict]:
        # Stage 1: BM25 Search
        bm25_retriever = ElasticsearchRetriever(
            es_client=self.es_client,
            index_name=self.index_name,
            body_func=self.bm25_query,
            content_field=self.text_field,
        )
        bm25_documents = bm25_retriever.invoke(search_query)

        # Stage 2: Semantic Search
        vector_retriever = ElasticsearchRetriever(
            es_client=self.es_client,
            index_name=self.index_name,
            body_func=self.vector_query,
            content_field=self.text_field
        )
        vector_documents = vector_retriever.invoke(search_query)

        # Combine BM25 and vector search results, remove duplicates based on document ID
        combined_documents = {doc.metadata['_id']: doc for doc in bm25_documents + vector_documents}
        combined_documents = list(combined_documents.values())
        print(f"Combined {len(bm25_documents)} BM25 and {len(vector_documents)} vector search results into {len(combined_documents)} unique documents.")
        # Stage 3: Neural Re-ranking
        pairs = [[search_query, doc.page_content] for doc in combined_documents]
        scores = self.reranker.compute_score(pairs)

        # Attach scores to documents and sort by scores
        for i, doc in enumerate(combined_documents):
            doc.metadata['score'] = scores[i]

        # Filter out documents with non-positive scores
        positive_documents = [doc for doc in combined_documents if doc.metadata['score'] > 0]
        sorted_documents = sorted(positive_documents, key=lambda x: x.metadata['score'], reverse=True)

        return sorted_documents

# Initialize ThreeStageSearch class
three_stage_search = ThreeStageSearch(es_client=es_client, embedder=embedder)

# Perform search
search_query = "KRAS mutation"
documents = three_stage_search.retrieve(search_query)

# Print the documents
for doc in documents:
    print(doc.page_content)


In [ ]:
len(documents)

In [ ]:
from typing import List, Dict
import logging
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from elasticsearch import Elasticsearch
from FlagEmbedding import FlagReranker

class ThreeStageSearch:
    def __init__(self, es_client: Elasticsearch, embedder) -> None:
        self.es_client = es_client
        self.embedder = embedder
        self.reranker = self.load_reranker()
        self.index_name = "eligibility_criteria"
        self.text_field = "criterion"

    def load_reranker(self) -> FlagReranker:
        return FlagReranker('BAAI/bge-reranker-large', use_fp16=True, device=5)

    def bm25_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
        return {
            "size": 8500,
            "query": {
                "bool": {
                    "must": [
                        {"match": {"eligibility_type.keyword": eligibility_type}},
                        {
                            "bool": {
                                "should": [
                                    {"match": {"criterion": {"query": search_query, "fuzziness": "AUTO"}}},
                                    {
                                        "nested": {
                                            "path": "entities",
                                            "query": {
                                                "bool": {
                                                    # "must": [{"match": {"entities.is_negated": "no"}}],
                                                    "should": [
                                                        {"match": {"entities.entity": {"query": search_query, "fuzziness": "AUTO"}}},
                                                        {"match": {"entities.synonyms": {"query": search_query, "fuzziness": "AUTO"}}}
                                                    ]
                                                }
                                            }
                                        }
                                    }
                                ]
                            }
                        }
                    ]
                }
            }
        }

    def vector_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
        vector = self.embedder.get_embeddings(search_query)
        return {
            "size": 8000,
            "query": {
                "bool": {
                    "must": [
                        {"match": {"eligibility_type.keyword": eligibility_type}}
                    ]
                }
            },
            "knn": {
                "field": "criterion_vector",
                "query_vector": vector,
                "num_candidates": 10000
            },
        }

    def reciprocal_rank_fusion(self, results: List[Dict], k: int = 60) -> List[Dict]:
        combined_scores = {}
        seen_docs = set()  # Set to track seen document IDs
        
        for result_set in results:
            source = result_set['source']
            for rank, result in enumerate(result_set['results']):
                doc_id = result['id']
                score = 1 / (k + rank + 1)  # RRF score
                if doc_id not in combined_scores:
                    combined_scores[doc_id] = 0
                combined_scores[doc_id] += score

        combined_results = [{'id': doc_id, 'score': score} for doc_id, score in combined_scores.items()]
        
        # Remove duplicates: This ensures that each document ID appears only once.
        unique_results = []
        seen_ids = set()
        for result in combined_results:
            if result['id'] not in seen_ids:
                unique_results.append(result)
                seen_ids.add(result['id'])
        
        unique_results.sort(key=lambda x: x['score'], reverse=True)
    
        return unique_results


    def retrieve(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> List[Dict]:
        # BM25 retrieval
        bm25_retriever = ElasticsearchRetriever(
            es_client=self.es_client,
            index_name=self.index_name,
            body_func=lambda query: self.bm25_query(query, eligibility_type),
            content_field=self.text_field,
        )
        bm25_documents = bm25_retriever.invoke(search_query)
        bm25_results = [{"id": doc.metadata['_id'], "score": doc.metadata.get('_score', 0)} for doc in bm25_documents]

        # Vector retrieval
        vector_retriever = ElasticsearchRetriever(
            es_client=self.es_client,
            index_name=self.index_name,
            body_func=lambda query: self.vector_query(query, eligibility_type),
            content_field=self.text_field,
        )
        vector_documents = vector_retriever.invoke(search_query)
        vector_results = [{"id": doc.metadata['_id'], "score": doc.metadata.get('_score', 0)} for doc in vector_documents]

        # Apply RRF
        rrf_results = self.reciprocal_rank_fusion([{"source": "bm25", "results": bm25_results}, {"source": "vector", "results": vector_results}])

        # Fetch documents based on RRF results
        rrf_doc_ids = [res['id'] for res in rrf_results]
        rrf_documents = []
        seen_docs = set()  # Set to track seen document IDs
        
        for doc in bm25_documents + vector_documents:
            if doc.metadata['_id'] in rrf_doc_ids and doc.metadata['_id'] not in seen_docs:
                rrf_documents.append(doc)
                seen_docs.add(doc.metadata['_id'])

        # Stage 3: Neural Re-ranking
        pairs = [[search_query, doc.page_content] for doc in rrf_documents]
        scores = self.reranker.compute_score(pairs)

        for i, doc in enumerate(rrf_documents):
            doc.metadata['score'] = scores[i]

        positive_documents = [doc for doc in rrf_documents if doc.metadata['score'] > 1]
        sorted_documents = sorted(positive_documents, key=lambda x: x.metadata['score'], reverse=True)

        return sorted_documents


# Initialize ThreeStageSearch class
three_stage_search = ThreeStageSearch(es_client=es_client, embedder=embedder)

# Perform search
search_query = "EGFR"
documents = three_stage_search.retrieve(search_query)

# Print the documents
for doc in documents:
    print(doc.page_content)


In [ ]:
len(documents)

In [ ]:
for doc in documents:
    print(doc.metadata["score"])

In [ ]:
documents[34].metadata["score"]

In [ ]:
from typing import List, Dict
import logging
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from elasticsearch import Elasticsearch
from FlagEmbedding import FlagReranker

class ThreeStageSearch:
    def __init__(self, es_client: Elasticsearch, embedder) -> None:
        self.es_client = es_client
        self.embedder = embedder
        self.reranker = self.load_reranker()
        self.index_name = "eligibility_criteria"
        self.text_field = "criterion"

    def load_reranker(self) -> FlagReranker:
        return FlagReranker('BAAI/bge-reranker-large', use_fp16=True, device=5)

    def bm25_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria", from_: int = 0) -> Dict:
        return {
            "from": from_,
            "size": 1000,  # Pagination size
            "query": {
                "bool": {
                    "must": [
                        {"match": {"eligibility_type.keyword": eligibility_type}},
                        {
                            "bool": {
                                "should": [
                                    {"match": {"criterion": {"query": search_query, "fuzziness": "AUTO"}}},
                                    {
                                        "nested": {
                                            "path": "entities",
                                            "query": {
                                                "bool": {
                                                    "must": [{"match": {"entities.is_negated": "no"}}],
                                                    "should": [
                                                        {"match": {"entities.entity": {"query": search_query, "fuzziness": "AUTO"}}},
                                                        {"match": {"entities.synonyms": {"query": search_query, "fuzziness": "AUTO"}}}
                                                    ]
                                                }
                                            }
                                        }
                                    }
                                ]
                            }
                        }
                    ]
                }
            }
        }

    def vector_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria", from_: int = 0) -> Dict:
        vector = self.embedder.get_embeddings(search_query)
        return {
            "from": from_,
            "size": 1000,  # Pagination size
            "query": {
                "bool": {
                    "must": [
                        {"match": {"eligibility_type.keyword": eligibility_type}}
                    ]
                }
            },
            "knn": {
                "field": "criterion_vector",
                "query_vector": vector,
                "num_candidates": 1000
            }
        }

    def reciprocal_rank_fusion(self, results: List[Dict], k: int = 30) -> List[Dict]:
        combined_scores = {}
        for result_set in results:
            source = result_set['source']
            for rank, result in enumerate(result_set['results']):
                doc_id = result['id']
                score = 1 / (k + rank + 1)  # RRF score
                if doc_id not in combined_scores:
                    combined_scores[doc_id] = 0
                combined_scores[doc_id] += score
        
        combined_results = [{'id': doc_id, 'score': score} for doc_id, score in combined_scores.items()]
        
        # Remove duplicates: This ensures that each document ID appears only once.
        unique_results = []
        seen_ids = set()
        for result in combined_results:
            if result['id'] not in seen_ids:
                unique_results.append(result)
                seen_ids.add(result['id'])
        
        unique_results.sort(key=lambda x: x['score'], reverse=True)
        
        return unique_results

    def retrieve(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> List[Dict]:
        page_size = 1000
        all_bm25_documents = []
        all_vector_documents = []

        # Paginate BM25 results
        for from_ in range(0, 9999, page_size):
            bm25_retriever = ElasticsearchRetriever(
                es_client=self.es_client,
                index_name=self.index_name,
                body_func=lambda query: self.bm25_query(query, eligibility_type, from_),
                content_field=self.text_field,
            )
            bm25_documents = bm25_retriever.invoke(search_query)
            all_bm25_documents.extend(bm25_documents)

        # Paginate Vector results
        for from_ in range(0, 9999, page_size):
            vector_retriever = ElasticsearchRetriever(
                es_client=self.es_client,
                index_name=self.index_name,
                body_func=lambda query: self.vector_query(query, eligibility_type, from_),
                content_field=self.text_field,
            )
            vector_documents = vector_retriever.invoke(search_query)
            all_vector_documents.extend(vector_documents)

        # Apply RRF
        bm25_results = [{"id": doc.metadata['_id'], "score": doc.metadata.get('_score', 0)} for doc in all_bm25_documents]
        vector_results = [{"id": doc.metadata['_id'], "score": doc.metadata.get('_score', 0)} for doc in all_vector_documents]
        rrf_results = self.reciprocal_rank_fusion([{"source": "bm25", "results": bm25_results}, {"source": "vector", "results": vector_results}])

        # Fetch documents based on RRF results
        rrf_doc_ids = [res['id'] for res in rrf_results]
        rrf_documents = []
        seen_docs = set()  # Set to track seen document IDs
        
        for doc in all_bm25_documents + all_vector_documents:
            if doc.metadata['_id'] in rrf_doc_ids and doc.metadata['_id'] not in seen_docs:
                rrf_documents.append(doc)
                seen_docs.add(doc.metadata['_id'])

        # Stage 3: Neural Re-ranking
        pairs = [[search_query, doc.page_content] for doc in rrf_documents]
        scores = self.reranker.compute_score(pairs)

        for i, doc in enumerate(rrf_documents):
            doc.metadata['score'] = scores[i]

        positive_documents = [doc for doc in rrf_documents if doc.metadata['score'] > 1]
        sorted_documents = sorted(positive_documents, key=lambda x: x.metadata['score'], reverse=True)

        return sorted_documents

# Initialize ThreeStageSearch class
three_stage_search = ThreeStageSearch(es_client=es_client, embedder=embedder)

# Perform search
search_query = "EGFR"
documents = three_stage_search.retrieve(search_query)

# Print the documents
for doc in documents:
    print(doc.page_content)


In [ ]:
for doc in documents:
    print(doc.metadata["_id"])

In [ ]:
import numpy as np
np.unique([doc.metadata["_id"] for doc in documents]).shape

In [ ]:

len(documents)

In [ ]:
from typing import List, Dict
import logging
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from elasticsearch import Elasticsearch
from FlagEmbedding import FlagReranker

class ThreeStageSearch:
    def __init__(self, es_client: Elasticsearch, embedder) -> None:
        self.es_client = es_client
        self.embedder = embedder
        self.reranker = self.load_reranker()
        self.index_name = "eligibility_criteria"
        self.text_field = "criterion"

    def load_reranker(self) -> FlagReranker:
        return FlagReranker('BAAI/bge-reranker-large', use_fp16=True, device=5)

    def bm25_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
        return {
            "size": 1000,  # Batch size
            "query": {
                "bool": {
                    "must": [
                        {"match": {"eligibility_type.keyword": eligibility_type}},
                        {
                            "bool": {
                                "should": [
                                    {"match": {"criterion": {"query": search_query, "fuzziness": "AUTO"}}},
                                    {
                                        "nested": {
                                            "path": "entities",
                                            "query": {
                                                "bool": {
                                                    "must": [{"match": {"entities.is_negated": "no"}}],
                                                    "should": [
                                                        {"match": {"entities.entity": {"query": search_query, "fuzziness": "AUTO"}}},
                                                        {"match": {"entities.synonyms": {"query": search_query, "fuzziness": "AUTO"}}}
                                                    ]
                                                }
                                            }
                                        }
                                    }
                                ]
                            }
                        }
                    ]
                }
            },
        }

    def vector_query(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> Dict:
        vector = self.embedder.get_embeddings(search_query)
        return {
            "size": 1000,  # Batch size
            "query": {
                "bool": {
                    "must": [
                        {"match": {"eligibility_type.keyword": eligibility_type}}
                    ]
                }
            },
            "knn": {
                "field": "criterion_vector",
                "query_vector": vector,
                "num_candidates": 1000
            },
        }

    def scroll(self, query: Dict, scroll_time: str = "3m") -> List[Dict]:
        response = self.es_client.search(index=self.index_name, body=query, scroll=scroll_time)
        scroll_id = response['_scroll_id']
        hits = response['hits']['hits']
        all_hits = hits

        while len(hits) > 0:
            response = self.es_client.scroll(scroll_id=scroll_id, scroll=scroll_time)
            hits = response['hits']['hits']
            all_hits.extend(hits)

        # Clear the scroll context
        self.es_client.clear_scroll(scroll_id=scroll_id)

        return all_hits

    def reciprocal_rank_fusion(self, results: List[Dict], k: int = 60) -> List[Dict]:
        combined_scores = {}
        for result_set in results:
            source = result_set['source']
            for rank, result in enumerate(result_set['results']):
                doc_id = result['id']
                score = 1 / (k + rank + 1)  # RRF score
                if doc_id not in combined_scores:
                    combined_scores[doc_id] = 0
                combined_scores[doc_id] += score
        
        combined_results = [{'id': doc_id, 'score': score} for doc_id, score in combined_scores.items()]
        
        # Remove duplicates: This ensures that each document ID appears only once.
        unique_results = []
        seen_ids = set()
        for result in combined_results:
            if result['id'] not in seen_ids:
                unique_results.append(result)
                seen_ids.add(result['id'])
        
        unique_results.sort(key=lambda x: x['score'], reverse=True)
        
        return unique_results

    def retrieve(self, search_query: str, eligibility_type: str = "Inclusion Criteria") -> List[Dict]:
        # Retrieve BM25 results using scroll
        bm25_query = self.bm25_query(search_query, eligibility_type)
        bm25_documents = self.scroll(bm25_query)

        # Retrieve Vector results using scroll
        vector_query = self.vector_query(search_query, eligibility_type)
        vector_documents = self.scroll(vector_query)

        # Apply RRF
        bm25_results = [{"id": doc['_id'], "score": doc['_score']} for doc in bm25_documents]
        vector_results = [{"id": doc['_id'], "score": doc['_score']} for doc in vector_documents]
        rrf_results = self.reciprocal_rank_fusion([{"source": "bm25", "results": bm25_results}, {"source": "vector", "results": vector_results}])

        # Fetch documents based on RRF results
        rrf_doc_ids = [res['id'] for res in rrf_results]
        rrf_documents = []
        seen_docs = set()  # Set to track seen document IDs
        
        for doc in bm25_documents + vector_documents:
            if doc['_id'] in rrf_doc_ids and doc['_id'] not in seen_docs:
                rrf_documents.append({
                    "id": doc['_id'],
                    "page_content": doc['_source'][self.text_field],
                    "metadata": doc['_source']
                })
                seen_docs.add(doc['_id'])

        # Stage 3: Neural Re-ranking
        pairs = [[search_query, doc['page_content']] for doc in rrf_documents]
        scores = self.reranker.compute_score(pairs)

        for i, doc in enumerate(rrf_documents):
            doc['metadata']['score'] = scores[i]

        positive_documents = [doc for doc in rrf_documents if doc['metadata']['score'] > 0]
        sorted_documents = sorted(positive_documents, key=lambda x: x['metadata']['score'], reverse=True)

        return sorted_documents

# Initialize ThreeStageSearch class
three_stage_search = ThreeStageSearch(es_client=es_client, embedder=embedder)

# Perform search
search_query = "EGFR"
documents = three_stage_search.retrieve(search_query)

# Print the documents
for doc in documents:
    print(doc['page_content'])

In [ ]:
import pickle

# Load the list from the pickle file
with open('documents.pkl', 'rb') as f:
    loaded_list = pickle.load(f)

# print(loaded_list)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from LM_Cocktail import mix_models, mix_models_with_data

tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-large')
model = mix_models(
    model_names_or_paths=["BAAI/bge-large-en-v1.5", "BAAI/bge-reranker-large"], 
    model_type='reranker', 
    weights=[0.5, 0.5],
    output_path="./mixed_reranker")

model.eval()

pairs = [['Bone lesions', 'bone fractures'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]
with torch.no_grad():
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
    scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
    print(scores)

In [ ]:
FlagReranker?

In [ ]:
for doc in loaded_list:
    print(doc.metadata["query"])

In [ ]:
embedder.get_embeddings("Adjuvant treatment")

In [ ]:
import pickle

# Load the list from the pickle file
with open('final_ranked_trials.pkl', 'rb') as f:
    loaded_list = pickle.load(f)


In [ ]:
len(loaded_list)

In [ ]:
loaded_list[0:10]

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from LM_Cocktail import mix_models, mix_models_with_data



def get_inputs(pairs, tokenizer, prompt=None, max_length=1024):
    if prompt is None:
        prompt = "Given a query (A) and a document (B), evaluate whether the document fulfills the intent of the query. Provide a prediction of 'Yes' if the document matches the query, and 'No' if it does not. Pay close attention to negations (e.g., 'not', 'no', 'none') in both the query and the document, as they may influence the match. Your prediction should strictly reflect whether the document satisfies the query's requirements."
    sep = "\n"
    prompt_inputs = tokenizer(prompt,
                              return_tensors=None,
                              add_special_tokens=False)['input_ids']
    sep_inputs = tokenizer(sep,
                           return_tensors=None,
                           add_special_tokens=False)['input_ids']
    inputs = []
    for query, passage in pairs:
        query_inputs = tokenizer(f'A: {query}',
                                 return_tensors=None,
                                 add_special_tokens=False,
                                 max_length=max_length * 3 // 4,
                                 truncation=True)
        passage_inputs = tokenizer(f'B: {passage}',
                                   return_tensors=None,
                                   add_special_tokens=False,
                                   max_length=max_length,
                                   truncation=True)
        item = tokenizer.prepare_for_model(
            [tokenizer.bos_token_id] + query_inputs['input_ids'],
            sep_inputs + passage_inputs['input_ids'],
            truncation='only_second',
            max_length=max_length,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            add_special_tokens=False
        )
        item['input_ids'] = item['input_ids'] + sep_inputs + prompt_inputs
        item['attention_mask'] = [1] * len(item['input_ids'])
        inputs.append(item)
    return tokenizer.pad(
            inputs,
            padding=True,
            max_length=max_length + len(sep_inputs) + len(prompt_inputs),
            pad_to_multiple_of=8,
            return_tensors='pt',
    )

tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-gemma')
model = AutoModelForCausalLM.from_pretrained('BAAI/bge-reranker-v2-gemma')
yes_loc = tokenizer('Yes', add_special_tokens=False)['input_ids'][0]
model.eval()

pairs =[['proteinuria at screening', 'No Proteinuria at screening as demonstrated by urine protein: urine creatinine (UPC) ratio of > or equal to 1.0 or urine dipstick for proteinuria > or equal to 2+ (patients discovered to have > or equal to 2+ proteinuria on dipstick urinalysis at baseline should have undergone a 24 hour urine collection and must have demonstrated < or equal to 1g of protein in 24 hours to be eligible).'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]
with torch.no_grad():
    inputs = get_inputs(pairs, tokenizer)
    scores = model(**inputs, return_dict=True).logits[:, -1, yes_loc].view(-1, ).float()
    print(scores)